In [1]:
# Cell 1：安装依赖（已安装可跳过）
%pip install -q torch transformers scikit-learn plotly pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# 自建小型新闻句库，按类别整理（内容为常见新闻报道示例句）
news = {
    '科技': [
        '华为发布新一代人工智能芯片，性能大幅提升。',
        'SpaceX 成功发射了可重复使用的运载火箭。',
        '中国科学家在量子计算领域取得重大突破。',
        '苹果公司推出了搭载自研芯片的新款笔记本电脑。',
        '特斯拉展示了全自动驾驶技术的最新进展。',
        '微软宣布开源多款人工智能大模型工具。',
    ],
    '体育': [
        '中国女排在世界杯决赛中逆转夺冠。',
        '梅西带领球队夺得世界冠军奖杯。',
        '北京冬奥会开幕式精彩纷呈，获得世界好评。',
        '苏炳添在百米短跑中刷新了亚洲纪录。',
        '国足新帅上任后首战取得胜利。',
        '全红婵在跳水比赛中再次斩获金牌。',
    ],
    '经济': [
        '央行宣布下调存款准备金率以支持实体经济。',
        'A股三大指数集体上涨，成交量明显放大。',
        '国家统计局发布最新经济数据，增速稳中有升。',
        '多家银行下调房贷利率，提振楼市信心。',
        '人民币汇率保持基本稳定，跨境资金流动平稳。',
        '新能源汽车出口量再创新高。',
    ],
    '娱乐': [
        '新上映的电影票房突破十亿元大关。',
        '知名歌手宣布将在全国举办巡回演唱会。',
        '真人版改编剧集上线后引发热议。',
        '某综艺节目收视率持续走高。',
        '实力派演员凭借新作获得最佳男主角奖。',
        '经典动画电影修复版重新上映。',
    ],
    '社会': [
        '多地上调最低工资标准，保障劳动者权益。',
        '暴雨天气导致部分城市出现内涝。',
        '社区开展义诊活动，居民纷纷点赞。',
        '警方破获一起电信网络诈骗案。',
        '大学生志愿服务队走进乡村支教。',
        '全国多地迎来降雪天气。',
    ],
}

rows = []
for category, sentences in news.items():
    for s in sentences:
        rows.append({'sentence': s, 'category': category})

df = pd.DataFrame(rows)
df

,sentence,category
0,华为发布新一代人工智能芯片，性能大幅提升。,科技
1,SpaceX 成功发射了可重复使用的运载火箭。,科技
2,中国科学家在量子计算领域取得重大突破。,科技
3,苹果公司推出了搭载自研芯片的新款笔记本电脑。,科技
4,特斯拉展示了全自动驾驶技术的最新进展。,科技
5,微软宣布开源多款人工智能大模型工具。,科技
6,中国女排在世界杯决赛中逆转夺冠。,体育
7,梅西带领球队夺得世界冠军奖杯。,体育
8,北京冬奥会开幕式精彩纷呈，获得世界好评。,体育
9,苏炳添在百米短跑中刷新了亚洲纪录。,体育


In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'bert-base-chinese'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()


def encode_sentences(sentences, batch_size=16):
    all_vectors = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors='pt',
        )
        ids = encoded['input_ids'].to(device)
        mask = encoded['attention_mask'].to(device)
        with torch.inference_mode():
            out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
            hidden = torch.stack(out.hidden_states[-4:]).mean(dim=0)  # (B, T, H) 取最后4层平均
        mask_exp = mask.unsqueeze(-1).float()  # (B, T, 1)
        # 对每个句子按 attention_mask 做 mean-pooling，得到固定维度句向量
        vec = (hidden * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)
        all_vectors.append(vec.cpu().numpy())
    return np.concatenate(all_vectors, axis=0)


embeddings = encode_sentences(df['sentence'].to_list())
print(f'Sentence embeddings shape: {embeddings.shape}')

d:\anaconda3\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence embeddings shape: (30, 768)


In [4]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

X = StandardScaler().fit_transform(embeddings)

# 样本量较小时，PCA 分量不能超过 min(n_samples, n_features)
X_pca = PCA(n_components=min(50, X.shape[0], X.shape[1]), random_state=SEED).fit_transform(X)

coords_3d = TSNE(
    n_components=3,
    perplexity=15,        # 样本量较小时调低
    learning_rate='auto',
    init='pca',
    max_iter=1500,
    random_state=SEED,
).fit_transform(X_pca)

plot_df = df.copy()
plot_df[['x', 'y', 'z']] = coords_3d
plot_df.head()

,sentence,category,x,y,z
0,华为发布新一代人工智能芯片，性能大幅提升。,科技,104.040894,-39.464405,-8.332057
1,SpaceX 成功发射了可重复使用的运载火箭。,科技,121.768448,72.148697,-149.079544
2,中国科学家在量子计算领域取得重大突破。,科技,59.396442,129.378143,-84.479225
3,苹果公司推出了搭载自研芯片的新款笔记本电脑。,科技,142.511673,-14.389256,-76.417091
4,特斯拉展示了全自动驾驶技术的最新进展。,科技,66.689972,51.765354,-24.799154


In [5]:
# 保存带坐标的结果
plot_df.to_csv('news_sentence_vectors.csv', index=False, encoding='utf-8-sig')

In [7]:
# Cell 6：Plotly 交互式三维可视化
import plotly.express as px

fig = px.scatter_3d(
    plot_df,
    x='x',
    y='y',
    z='z',
    color='category',
    hover_name='sentence',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title='Chinese BERT 新闻句向量三维可视化',
)

fig.update_traces(marker=dict(size=4.5, opacity=0.80, line=dict(width=0.3, color='white')))
fig.update_layout(
    width=1000,
    height=800,
    legend_title_text='新闻类别',
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='rgb(248, 249, 252)',
    ),
    paper_bgcolor='white',
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.show()
fig.write_html('news_sentence_vec_3d.html')